# 🧪 Sesión 23 · Operaciones CRUD Transaccionales y Async Sessions

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Por qué esperar a la base de datos no debe bloquear el servidor?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/libros  {"titulo":"Rayuela","anio":1963}  -> 201
GET  /api/libros                                  -> 200
GET  /api/libros/1                                -> 200
PUT  /api/libros/1  {"titulo":"Rayuela (edición)"} -> 200
DELETE /api/libros/1                              -> 204
GET  /api/libros/1                                -> 404
```

_La base de datos sigue siendo SQLite; lo que cambia es cómo la aplicación espera sus respuestas._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Implementar un CRUD con SQLAlchemy asíncrono.
- Usar AsyncSession y aiosqlite.
- Verificar 201, 200, 404 y 204 en el mismo flujo.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] Crear un libro → HTTP 201
- [ ] Listar los libros → HTTP 200
- [ ] Actualizar el libro → HTTP 200
- [ ] Eliminar el libro → HTTP 204
- [ ] El libro eliminado ya no existe → HTTP 404
- [ ] La persistencia es asíncrona

## 🧩 El reto

Implementa un CRUD completo con SQLAlchemy asíncrono y aiosqlite. El servidor no debe bloquearse mientras espera la base.

**Requisitos del contrato**

- Usa create_async_engine con sqlite+aiosqlite.
- Cada endpoint usa AsyncSession y await.
- POST responde 201; GET 200; PUT 200; DELETE 204.
- Un id inexistente responde 404.
- La tabla se crea al iniciar la aplicación.

> **Pista conceptual:** Si tu endpoint es `def` y usas una sesión síncrona, estás bloqueando el event loop. La regla es: motor async, sesión async, endpoint async.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion23.md` con la diferencia entre def y async def.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [13]:
%%writefile app.py
import asyncio
import os
from fastapi import FastAPI, HTTPException, status, Depends
from pydantic import BaseModel
from sqlalchemy.ext.asyncio import create_async_engine, async_sessionmaker, AsyncSession
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column
from sqlalchemy import select

DATABASE_URL = "sqlite+aiosqlite:///./biblioteca.db"

engine = create_async_engine(
    DATABASE_URL,
    echo=False,
    connect_args={"check_same_thread": False}
)
async_session = async_sessionmaker(engine, expire_on_commit=False, class_=AsyncSession)

class Base(DeclarativeBase):
    pass

class LibroDB(Base):
    __tablename__ = "libros"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    titulo: Mapped[str] = mapped_column(nullable=False)
    anio: Mapped[int] = mapped_column(nullable=False)

class LibroCreate(BaseModel):
    titulo: str
    anio: int

class LibroResponse(BaseModel):
    id: int
    titulo: str
    anio: int

    class Config:
        from_attributes = True

class ListaLibrosResponse(BaseModel):
    libros: list[LibroResponse]

app = FastAPI(title="CRUD Asíncrono de Libros")

# Esta función garantiza la creación dinámica de la tabla si no existe al recibir cada petición
async def asegurar_tablas():
    async with engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)

async def get_db():
    await asegurar_tablas()
    async with async_session() as session:
        try:
            yield session
        finally:
            await session.close()

@app.post("/api/libros", response_model=LibroResponse, status_code=status.HTTP_201_CREATED)
async def crear_libro(libro: LibroCreate, db: AsyncSession = Depends(get_db)):
    nuevo_libro = LibroDB(titulo=libro.titulo, anio=libro.anio)
    db.add(nuevo_libro)
    await db.commit()
    await db.refresh(nuevo_libro)
    return nuevo_libro

@app.get("/api/libros", response_model=ListaLibrosResponse)
async def listar_libros(db: AsyncSession = Depends(get_db)):
    result = await db.execute(select(LibroDB))
    libros = result.scalars().all()
    return {"libros": list(libros)}

@app.get("/api/libros/{libro_id}", response_model=LibroResponse)
async def obtener_libro(libro_id: int, db: AsyncSession = Depends(get_db)):
    result = await db.execute(select(LibroDB).where(LibroDB.id == libro_id))
    libro = result.scalar_one_or_none()
    if libro is None:
        raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail="Libro no encontrado")
    return libro

@app.put("/api/libros/{libro_id}", response_model=LibroResponse)
async def actualizar_libro(libro_id: int, libro_data: LibroCreate, db: AsyncSession = Depends(get_db)):
    result = await db.execute(select(LibroDB).where(LibroDB.id == libro_id))
    libro = result.scalar_one_or_none()
    if libro is None:
        raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail="Libro no encontrado")
    libro.titulo = libro_data.titulo
    libro.anio = libro_data.anio
    await db.commit()
    await db.refresh(libro)
    return libro

@app.delete("/api/libros/{libro_id}", status_code=status.HTTP_204_NO_CONTENT)
async def eliminar_libro(libro_id: int, db: AsyncSession = Depends(get_db)):
    result = await db.execute(select(LibroDB).where(LibroDB.id == libro_id))
    libro = result.scalar_one_or_none()
    if libro is None:
        raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail="Libro no encontrado")
    await db.delete(libro)
    await db.commit()
    return

Overwriting app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [14]:
from verificar import probar_http

casos = [{'descripcion': 'Crear un libro',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'Rayuela', 'anio': 1963},
  'estado': 201,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela'}},
 {'descripcion': 'Listar los libros',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'estado': 200,
  'json_contiene': {'libros': [{'id': 1}]}},
 {'descripcion': 'Actualizar el libro',
  'metodo': 'PUT',
  'ruta': '/api/libros/1',
  'json': {'titulo': 'Rayuela (edición crítica)', 'anio': 1963},
  'estado': 200,
  'json_contiene': {'id': 1, 'titulo': 'Rayuela (edición crítica)'}},
 {'descripcion': 'Eliminar el libro', 'metodo': 'DELETE', 'ruta': '/api/libros/1', 'estado': 204},
 {'descripcion': 'El libro eliminado ya no existe', 'metodo': 'GET', 'ruta': '/api/libros/1', 'estado': 404}]

probar_http(casos, archivo="app.py", limpiar=['biblioteca.db'], env={})


VERIFICACIÓN DEL RESULTADO · 5 casos · app.py
✅ 01. Crear un libro
✅ 02. Listar los libros
✅ 03. Actualizar el libro
✅ 04. Eliminar el libro
✅ 05. El libro eliminado ya no existe
------------------------------------------------------------------------------
RESULTADO: 5/5 casos correctos. ¡Resultado logrado!


In [15]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", regex=["AsyncSession|async_sessionmaker|aiosqlite"], descripcion="La persistencia es asíncrona"))
resumen(resultados)


✅ La persistencia es asíncrona
------------------------------------------------------------------------------
RESULTADO: 1/1 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion23.md` a `auditorias/auditoria_sesion23.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
